**Cell 1: Import Libraries and Load Merged Features**  
This cell loads the merged customer feature dataset and imports the statistical packages needed for hypothesis testing.  
- **Purpose**: Prepare the notebook environment and read the merged feature table.  
- **Key libraries**: `pandas`, `scipy.stats`, `statsmodels.stats.multitest`.  
- **No hypothesis testing here** — this is data setup.  
- **Expected output**: preview of the merged feature dataset.

In [3]:
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../../data/04_merged/final_merged_features.csv")

df.head()

,co_ref,total_spent,avg_payment,num_payments,last_payment_date,cutoff_date,days_since_last_payment,payments_last_30,payments_last_90,spend_last_30,...,days_since_last_call,calls_last_7,calls_last_14,calls_last_30,serious_complaints,switch_intent,cancel_intent,price_discussions,discount_requests,recent_call_ratio
0,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA0794,2927,975.666667,3,2025-01-10,2024-06-26,-198,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA0794,2927,975.666667,3,2025-01-10,2024-06-26,-198,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Cell 2: Create Binary Churn Target**  
This cell converts `prospect_outcome` into a binary target, where 1 means churned and 0 means retained.  
- **Purpose**: Define the dependent variable used in all subsequent hypothesis tests.  
- **No hypothesis testing here** — this is target creation.  
- **Expected output**: churn distribution counts.

In [4]:
df["target"] = (df["prospect_outcome"] == "Churned").astype(int)

df["target"].value_counts()

target
0    102310
1     15011
Name: count, dtype: int64

**Cell 3: Define Feature Sets Across All Sources**  
This cell lists billings, CC calls, emails, and renewal call features to test against churn.  
- **Purpose**: Specify the full merged feature set for hypothesis testing.  
- **No hypothesis testing here** — this is feature selection.  
- **Expected output**: feature lists stored for analysis.

In [5]:
num_cols = [
    # Billings features
    "total_spent", "avg_payment", "num_payments", "days_since_last_payment",
    "payments_last_30", "payments_last_90", "spend_last_30", "spend_last_90",
    "payment_trend", "tenure_years",
    # CC calls features
    "total_cc_calls", "days_since_last_cc_call", "cc_calls_last_7", "cc_calls_last_30",
    "total_complaints", "customer_issues", "financial_issues", "pricing_mentions",
    "avg_cc_sentiment", "repeat_call_ratio", "high_call_volume",
    # Email features
    "total_interactions", "interactions_14_days", "pre_renewal_interactions",
    "last_moment_engagement_ratio", "complaints", "negative_experience",
    "support_issues", "financial_stress", "price_dissatisfaction", "avg_sentiment",
    "agent_followups", "engagement_score",
    # Renewal calls features
    "total_calls", "days_since_last_call", "calls_last_7", "calls_last_14",
    "calls_last_30", "serious_complaints", "switch_intent", "cancel_intent",
    "price_discussions", "discount_requests", "recent_call_ratio"
]

cat_cols = [
    "payment_method_mode"
]

**Cell 4: Run Numerical Hypothesis Tests**  
This cell performs Mann-Whitney U tests for each numerical merged feature.  
- **Null Hypothesis (H₀)**: The feature distribution is the same for churn and non-churn groups.  
- **Alternative Hypothesis (H₁)**: The feature distribution differs between churn and non-churn groups.  
- **Test used**: Mann-Whitney U (non-parametric).  
- **Expected output**: summary table with p-values for each numerical feature.

In [6]:
results = []

for col in num_cols:
    churn = df[df["target"] == 1][col].dropna()
    non_churn = df[df["target"] == 0][col].dropna()

    stat, p_value = mannwhitneyu(churn, non_churn)

    results.append({
        "feature": col,
        "p_value": p_value
    })

num_results = pd.DataFrame(results)
num_results

,feature,p_value
0,total_spent,0.000000e+00
1,avg_payment,0.000000e+00
2,num_payments,0.000000e+00
3,days_since_last_payment,0.000000e+00
4,payments_last_30,5.896370e-02
5,payments_last_90,2.021271e-32
6,spend_last_30,6.442269e-02
7,spend_last_90,2.754201e-34
8,payment_trend,1.000000e+00
9,tenure_years,0.000000e+00


**Cell 5: Correct for Multiple Testing**  
This cell applies the Benjamini-Hochberg procedure to adjust the p-values from the numerical tests.  
- **Purpose**: Control the false discovery rate when testing many features.  
- **Key output**: `adjusted_p` values and a `significant` indicator.  
- **Decision rule**: A feature is significant if `adjusted_p < 0.05`.

In [7]:
p_values = num_results["p_value"]

adjusted_p = multipletests(p_values, method="fdr_bh")[1]

num_results["adjusted_p"] = adjusted_p
num_results["significant"] = adjusted_p < 0.05

num_results = num_results.sort_values("adjusted_p")

num_results

,feature,p_value,adjusted_p,significant
0,total_spent,0.000000e+00,0.000000e+00,True
1,avg_payment,0.000000e+00,0.000000e+00,True
2,num_payments,0.000000e+00,0.000000e+00,True
3,days_since_last_payment,0.000000e+00,0.000000e+00,True
9,tenure_years,0.000000e+00,0.000000e+00,True
28,financial_stress,0.000000e+00,0.000000e+00,True
30,avg_sentiment,0.000000e+00,0.000000e+00,True
34,days_since_last_call,0.000000e+00,0.000000e+00,True
40,cancel_intent,0.000000e+00,0.000000e+00,True
26,negative_experience,2.326647e-296,1.023725e-295,True


**Cell 6: Test Categorical Feature Association**  
This cell checks whether the categorical feature `payment_method_mode` is associated with churn using a chi-square test.  
- **Null Hypothesis (H₀)**: `payment_method_mode` is independent of churn.  
- **Alternative Hypothesis (H₁)**: `payment_method_mode` is associated with churn.  
- **Test used**: Chi-square test of independence.  
- **Expected output**: a DataFrame of p-values for categorical features.

In [8]:
cat_results = []

for col in cat_cols:
    table = pd.crosstab(df[col], df["target"])
    chi2, p_value, _, _ = chi2_contingency(table)

    cat_results.append({
        "feature": col,
        "p_value": p_value
    })

cat_results = pd.DataFrame(cat_results)
cat_results

,feature,p_value
0,payment_method_mode,0.0


**Cell 7: Save Merged Feature Validation Results**  
This cell writes the numerical and categorical hypothesis test results to CSV files for reporting.  
- **Purpose**: Persist the merged feature validation outcomes.  
- **No hypothesis testing here** — this is the output step.  
- **Expected output**: saved reports in the `reports` folder.

In [9]:
num_results.to_csv(
    "../../reports/merged_feature_validation_results.csv",
    index=False
)

cat_results.to_csv(
    "../../reports/merged_feature_validation_categorical_results.csv",
    index=False
)